# 🎓 Proyecto Final — Modelos de Clasificación en Machine Learning

| | |
|---|---|
| **Tema** | Predicción de tipo de personalidad MBTI mediante modelos supervisados |
| **Dataset** | Encuesta de 60 preguntas tipo Likert — 59,999 registros, 16 clases |
| **Modelos** | Árbol de Decisión · Bagging · Random Forest (GridSearchCV) |

---
## Índice
1. [Análisis Inicial de Datos](#analisis)
2. [Conversión de Datos](#conversion)
3. [División de Datos](#division)
4. [Modelo de Árbol de Clasificación](#arbol)
5. [Modelo de Bagging](#bagging)
6. [Modelo Random Forest + GridSearchCV](#rf)
7. [Análisis y Conclusiones](#conclusiones)
---

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELDA 0 · Importaciones y configuración global
# ═══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

from sklearn.model_selection  import train_test_split, GridSearchCV
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import BaggingClassifier, RandomForestClassifier
from sklearn.metrics          import (accuracy_score, classification_report,
                                       confusion_matrix, ConfusionMatrixDisplay)

# ── Tema oscuro global ───────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0D0D0D',
    'axes.facecolor'   : '#0D0D0D',
    'text.color'       : '#EEEEEE',
    'axes.labelcolor'  : '#EEEEEE',
    'xtick.color'      : '#CCCCCC',
    'ytick.color'      : '#CCCCCC',
    'axes.edgecolor'   : '#333333',
    'grid.color'       : '#2A2A2A',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.6,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

# ── Paleta MBTI por grupo temperamental ─────────────────────────────────────
PALETA = {
    'INTJ':'#7B5FCC','INTP':'#9B8DC4','ENTJ':'#B8AED8','ENTP':'#D4CAEC',  # Analistas
    'INFJ':'#2ECC71','INFP':'#58D68D','ENFJ':'#82E0AA','ENFP':'#ABEBC6',  # Diplomáticos
    'ISTJ':'#3498DB','ISFJ':'#5DADE2','ESTJ':'#85C1E9','ESFJ':'#AED6F1',  # Centinelas
    'ISTP':'#E67E22','ISFP':'#EF9A3A','ESTP':'#F5B95A','ESFP':'#FAD59A',  # Exploradores
}
COLORES_GRUPO = {'Analistas':'#7B5FCC','Diplomáticos':'#2ECC71',
                 'Centinelas':'#3498DB','Exploradores':'#E67E22'}

print('✔  Librerías cargadas | Tema oscuro activado')

<a id='analisis'></a>
---
# 📂 PARTE I · Análisis Inicial de Datos

## 1.1 · Carga del dataset
Se utiliza `pandas.read_csv()` con `encoding='latin1'` para manejar correctamente los caracteres especiales del archivo original.

In [ ]:
df_raw = pd.read_csv('Datos_pr.csv', encoding='latin1')

print(f'✔  Archivo cargado exitosamente')
print(f'   Filas    : {df_raw.shape[0]:,}')
print(f'   Columnas : {df_raw.shape[1]}')
print(f'   Memoria  : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB\n')
df_raw.head(3)

## 1.2 · Preguntas de la encuesta (idioma original — inglés)

In [ ]:
preguntas_en = [c for c in df_raw.columns if c not in ('Response Id', 'Personality')]
print(f'Total de preguntas: {len(preguntas_en)}\n')
for i, q in enumerate(preguntas_en, 1):
    print(f'  Q{i:02d}: {q}')

## 1.3 · Traducción de columnas al español
Se construye un diccionario de mapeo inglés → español para renombrar todas las columnas del dataframe.

In [ ]:
TRADUCCION = {
    'Response Id': 'ID_Respuesta',
    'You regularly make new friends.': 'Haces nuevos amigos con regularidad.',
    'You spend a lot of your free time exploring various random topics that pique your interest':
        'Dedicas mucho tiempo libre a explorar temas variados que despiertan tu curiosidad.',
    'Seeing other people cry can easily make you feel like you want to cry too':
        'Ver a otras personas llorar fácilmente te hace querer llorar también.',
    'You often make a backup plan for a backup plan.':
        'A menudo haces un plan de respaldo para tu plan de respaldo.',
    'You usually stay calm, even under a lot of pressure':
        'Generalmente te mantienes calmado/a incluso bajo mucha presión.',
    'At social events, you rarely try to introduce yourself to new people and mostly talk to the ones you already know':
        'En eventos sociales, rara vez intentas presentarte con personas nuevas.',
    'You prefer to completely finish one project before starting another.':
        'Prefieres terminar completamente un proyecto antes de comenzar otro.',
    'You are very sentimental.':
        'Eres muy sentimental.',
    'You like to use organizing tools like schedules and lists.':
        'Te gusta usar herramientas de organización como agendas y listas.',
    'Even a small mistake can cause you to doubt your overall abilities and knowledge.':
        'Incluso un pequeño error puede hacerte dudar de tus habilidades y conocimientos.',
    'You feel comfortable just walking up to someone you find interesting and striking up a conversation.':
        'Te sientes cómodo/a acercándote a alguien interesante para entablar conversación.',
    'You are not too interested in discussing various interpretations and analyses of creative works.':
        'No te interesa mucho discutir interpretaciones de obras creativas.',
    'You are more inclined to follow your head than your heart.':
        'Te inclinas más a seguir tu cabeza que tu corazón.',
    'You usually prefer just doing what you feel like at any given moment instead of planning a particular daily routine.':
        'Prefieres hacer lo que se te antoja en el momento en lugar de planificar una rutina.',
    'You rarely worry about whether you make a good impression on people you meet.':
        'Rara vez te preocupa si causas buena impresión en las personas que conoces.',
    'You enjoy participating in group activities.':
        'Disfrutas participar en actividades grupales.',
    'You like books and movies that make you come up with your own interpretation of the ending.':
        'Te gustan libros y películas que te invitan a crear tu propia interpretación del final.',
    'Your happiness comes more from helping others accomplish things than your own accomplishments.':
        'Tu felicidad viene más de ayudar a otros que de tus propios logros.',
    'You are interested in so many things that you find it difficult to choose what to try next.':
        'Te interesan tantas cosas que te resulta difícil elegir qué intentar a continuación.',
    'You are prone to worrying that things will take a turn for the worse.':
        'Tiendes a preocuparte de que las cosas empeoren.',
    'You avoid leadership roles in group settings.':
        'Evitas los roles de liderazgo en entornos grupales.',
    'You are definitely not an artistic type of person.':
        'Definitivamente no eres una persona artística.',
    'You think the world would be a better place if people relied more on rationality and less on their feelings.':
        'Crees que el mundo sería mejor si las personas confiaran más en la razón y menos en sus emociones.',
    'You prefer to do your chores before allowing yourself to relax.':
        'Prefieres hacer tus tareas antes de permitirte relajar.',
    'You enjoy watching people argue.':
        'Disfrutas ver a las personas discutir.',
    'You tend to avoid drawing attention to yourself.':
        'Tiendes a evitar llamar la atención sobre ti mismo/a.',
    'Your mood can change very quickly.':
        'Tu estado de ánimo puede cambiar muy rápidamente.',
    'You lose patience with people who are not as efficient as you.':
        'Pierdes la paciencia con personas que no son tan eficientes como tú.',
    'You often end up doing things at the last possible moment.':
        'A menudo terminas haciendo las cosas en el último momento posible.',
    'You have always been fascinated by the question of what, if anything, happens after death.':
        'Siempre te ha fascinado la pregunta de qué sucede después de la muerte.',
    'You usually prefer to be around others rather than on your own.':
        'Generalmente prefieres estar rodeado/a de otros antes que estar solo/a.',
    'You become bored or lose interest when the discussion gets highly theoretical.':
        'Te aburres cuando la discusión se vuelve muy teórica.',
    'You find it easy to empathize with a person whose experiences are very different from yours.':
        'Te resulta fácil empatizar con personas cuyas experiencias son muy distintas a las tuyas.',
    'You usually postpone finalizing decisions for as long as possible.':
        'Generalmente pospones la toma de decisiones el mayor tiempo posible.',
    'You rarely second-guess the choices that you have made.':
        'Rara vez cuestionas las decisiones que has tomado.',
    'After a long and exhausting week, a lively social event is just what you need.':
        'Después de una semana agotadora, un evento social animado es justo lo que necesitas.',
    'You enjoy going to art museums.':
        'Disfrutas visitar museos de arte.',
    'You often have a hard time understanding other people\x92s feelings.':
        'A menudo te cuesta entender los sentimientos de otras personas.',
    'You like to have a to-do list for each day.':
        'Te gusta tener una lista de tareas para cada día.',
    'You rarely feel insecure.':
        'Rara vez te sientes inseguro/a.',
    'You avoid making phone calls.':
        'Evitas hacer llamadas telefónicas.',
    'You often spend a lot of time trying to understand views that are very different from your own.':
        'A menudo dedicas tiempo a comprender puntos de vista muy diferentes a los tuyos.',
    'In your social circle, you are often the one who contacts your friends and initiates activities.':
        'En tu círculo social, eres quien contacta a los amigos e inicia actividades.',
    'If your plans are interrupted, your top priority is to get back on track as soon as possible.':
        'Si tus planes se interrumpen, tu prioridad es retomar el rumbo lo antes posible.',
    'You are still bothered by mistakes that you made a long time ago.':
        'Aún te molestan los errores que cometiste hace mucho tiempo.',
    'You rarely contemplate the reasons for human existence or the meaning of life.':
        'Rara vez contemplas las razones de la existencia humana o el significado de la vida.',
    'Your emotions control you more than you control them.':
        'Tus emociones te controlan más de lo que tú las controlas.',
    'You take great care not to make people look bad, even when it is completely their fault.':
        'Tienes cuidado de no hacer quedar mal a las personas, incluso cuando es su culpa.',
    'Your personal work style is closer to spontaneous bursts of energy than organized and consistent efforts.':
        'Tu estilo de trabajo es más de ráfagas espontáneas que de esfuerzos organizados y constantes.',
    'When someone thinks highly of you, you wonder how long it will take them to feel disappointed in you.':
        'Cuando alguien piensa bien de ti, te preguntas cuánto tardará en decepcionarse.',
    'You would love a job that requires you to work alone most of the time.':
        'Te encantaría un trabajo que requiera trabajar solo/a la mayor parte del tiempo.',
    'You believe that pondering abstract philosophical questions is a waste of time.':
        'Crees que reflexionar sobre preguntas filosóficas abstractas es una pérdida de tiempo.',
    'You feel more drawn to places with busy, bustling atmospheres than quiet, intimate places.':
        'Te atraen más los lugares concurridos y animados que los tranquilos e íntimos.',
    'You know at first glance how someone is feeling.':
        'Sabes a primera vista cómo se siente alguien.',
    'You often feel overwhelmed.':
        'A menudo te sientes abrumado/a.',
    'You complete things methodically without skipping over any steps.':
        'Completas las cosas de forma metódica sin saltarte ningún paso.',
    'You are very intrigued by things labeled as controversial.':
        'Te intrigan mucho las cosas catalogadas como controvertidas.',
    'You would pass along a good opportunity if you thought someone else needed it more.':
        'Cederías una buena oportunidad si creyeras que alguien más la necesita.',
    'You struggle with deadlines.':
        'Te cuesta cumplir con los plazos.',
    'You feel confident that things will work out for you.':
        'Te sientes seguro/a de que las cosas saldrán bien para ti.',
    'Personality': 'Personalidad',
}

df = df_raw.rename(columns=TRADUCCION)
# Limpieza defensiva de BOM (compatibilidad Windows)
df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()

print('✔  Columnas traducidas al español')
print(f'   Ejemplo: "{preguntas_en[0][:45]}..."')
print(f'         → "{list(TRADUCCION.values())[1][:45]}..."\n')
df.head(3)

## 1.4 · Guardar dataset traducido

In [ ]:
df.to_csv('datos_trad.csv', index=False, encoding='utf-8-sig')
print('✔  Archivo "datos_trad.csv" guardado en el directorio actual')

## 1.5 · Número de registros

In [ ]:
COL_ID     = df.columns[0]
COL_TARGET = df.columns[-1]

resumen = pd.DataFrame({
    'Métrica'  : ['Registros totales','Total columnas','Preguntas encuesta','Columnas de control'],
    'Valor'    : [f"{len(df):,}", str(df.shape[1]), str(df.shape[1]-2), f'{COL_ID}, {COL_TARGET}'],
})
print(f'Columna ID      → {COL_ID}')
print(f'Columna objetivo → {COL_TARGET}\n')
resumen

## 1.6 · Distribución por tipo de personalidad

In [ ]:
conteo = df[COL_TARGET].value_counts().sort_values(ascending=False)
colores_bar = [PALETA.get(p, '#888') for p in conteo.index]

fig, ax = plt.subplots(figsize=(15, 6))
bars = ax.bar(conteo.index, conteo.values, color=colores_bar,
              edgecolor='#1A1A1A', linewidth=0.8, width=0.72, zorder=3)

for bar, val in zip(bars, conteo.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 18,
            f'{val:,}', ha='center', va='bottom', fontsize=8.5,
            fontweight='bold', color='#EEEEEE')

media = conteo.mean()
ax.axhline(media, color='#E74C3C', lw=1.5, ls='--', zorder=4)

leyenda = [
    mpatches.Patch(color='#7B5FCC', label='Analistas  (NT)'),
    mpatches.Patch(color='#2ECC71', label='Diplomáticos (NF)'),
    mpatches.Patch(color='#3498DB', label='Centinelas (SJ)'),
    mpatches.Patch(color='#E67E22', label='Exploradores (SP)'),
    mpatches.Patch(color='#E74C3C', label=f'Media: {media:,.0f}'),
]
ax.legend(handles=leyenda, loc='lower right', fontsize=9,
          facecolor='#1A1A1A', edgecolor='#444444')

ax.set_title('Distribución de Tipos de Personalidad MBTI', fontsize=14,
             fontweight='bold', pad=14)
ax.set_xlabel('Tipo de Personalidad', fontsize=11, labelpad=8)
ax.set_ylabel('Número de Personas', fontsize=11, labelpad=8)
ax.set_ylim(0, conteo.max() * 1.13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.yaxis.grid(True, zorder=0)
plt.tight_layout()
plt.savefig('grafica_personalidades.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 1.7 · Grupos de personalidad identificados

In [ ]:
INFO_MBTI = {
    'INTJ':('El Arquitecto',   'Analistas',    'Estratega independiente e imaginativo. Planificador a largo plazo, perfeccionista y lógico.'),
    'INTP':('El Lógico',       'Analistas',    'Analítico e innovador. Disfruta resolver problemas complejos y explorar teorías abstractas.'),
    'ENTJ':('El Comandante',   'Analistas',    'Líder nato y asertivo. Toma decisiones rápidas y dirige equipos con eficiencia y visión.'),
    'ENTP':('El Innovador',    'Analistas',    'Creativo y debatidor. Le encanta cuestionar ideas y encontrar soluciones originales.'),
    'INFJ':('El Defensor',     'Diplomáticos', 'Idealista, empático y reservado. Lucha por sus valores con determinación y visión clara.'),
    'INFP':('El Mediador',     'Diplomáticos', 'Soñador, empático y fiel a sus principios. Valora la autenticidad y el sentido profundo.'),
    'ENFJ':('El Protagonista', 'Diplomáticos', 'Carismático y orientado a personas. Inspira a los demás y gestiona emociones con maestría.'),
    'ENFP':('El Activista',    'Diplomáticos', 'Entusiasta y sociable. Ve el potencial en cada persona y conecta ideas con emoción.'),
    'ISTJ':('El Logístico',    'Centinelas',   'Responsable, ordenado y confiable. Detallista, cumple siempre con sus compromisos.'),
    'ISFJ':('El Protector',    'Centinelas',   'Cálido y servicial. Se preocupa por el bienestar ajeno y es fiel a sus tradiciones.'),
    'ESTJ':('El Ejecutivo',    'Centinelas',   'Práctico y organizador. Valora el orden y la eficiencia; excelente administrador.'),
    'ESFJ':('El Cónsul',       'Centinelas',   'Sociable y solícito. Busca la armonía grupal y es muy sensible a las necesidades ajenas.'),
    'ISTP':('El Virtuoso',     'Exploradores', 'Observador y práctico. Le fascina entender cómo funcionan las cosas y resolver problemas técnicos.'),
    'ISFP':('El Aventurero',   'Exploradores', 'Artístico y sensible. Vive el presente con intensidad y expresa su creatividad libremente.'),
    'ESTP':('El Empresario',   'Exploradores', 'Energético y audaz. Actúa antes de pensar, disfruta el riesgo y lee muy bien a las personas.'),
    'ESFP':('El Animador',     'Exploradores', 'Espontáneo y divertido. Ama ser el centro de atención y disfruta la vida al máximo.'),
}

filas = []
for tipo in conteo.index:
    nombre, grupo, desc = INFO_MBTI[tipo]
    filas.append({'Tipo':tipo,'Nombre':nombre,'Grupo':grupo,
                  'Descripción':desc,'Registros':f"{conteo[tipo]:,}"})

tabla_tipos = pd.DataFrame(filas)

GRUPO_COLOR = {'Analistas':'#1E1040','Diplomáticos':'#0D2B1A',
               'Centinelas':'#0D1E33','Exploradores':'#2B1A06'}

def color_grupo(row):
    c = GRUPO_COLOR.get(row['Grupo'], '#111')
    return [f'background-color:{c};color:#EEE'] * len(row)

tabla_tipos.style.apply(color_grupo, axis=1).hide(axis='index')

<a id='conversion'></a>
---
# 🔢 PARTE II · Conversión de Datos

## 2.1 · Convertir etiquetas categóricas a numéricas con `pd.factorize()`

`pd.factorize()` asigna un entero único a cada categoría en el orden en que aparece, devolviendo además el array de clases originales para poder recuperar las etiquetas después.

In [ ]:
# Features y variable objetivo
X     = df.drop(columns=[COL_ID, COL_TARGET])
y_cat = df[COL_TARGET]

# Conversión categórica → numérica
y, CLASES = pd.factorize(y_cat)

# Mapa de referencia código ↔ personalidad
mapa = pd.DataFrame({'Código':range(len(CLASES)), 'Tipo de Personalidad':CLASES})

print('✔  pd.factorize() aplicado correctamente')
print(f'   Features (X)     : {X.shape[1]} columnas × {X.shape[0]:,} registros')
print(f'   Variable objetivo: {len(CLASES)} clases únicas')
print(f'   Ejemplo          : "{y_cat.iloc[0]}" → {y[0]}\n')
mapa

<a id='division'></a>
---
# ✂️ PARTE III · División de Datos

## 3.1 · Separar en conjuntos de entrenamiento (80%) y prueba (20%)

Se usa `stratify=y` para garantizar que la proporción de las 16 clases sea idéntica en ambos conjuntos, evitando sesgos en la evaluación.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y        # proporción de clases conservada en ambos conjuntos
)

resumen_split = pd.DataFrame({
    'Conjunto'   : ['Entrenamiento', 'Prueba', 'Total'],
    'Registros'  : [f'{len(X_train):,}', f'{len(X_test):,}', f'{len(X):,}'],
    'Proporción' : ['80 %', '20 %', '100 %'],
})

print('✔  División completada con estratificación')
print(f'   random_state=42 → reproducible en cualquier ejecución\n')
resumen_split

<a id='arbol'></a>
---
# 🌳 PARTE IV · Modelo de Árbol de Clasificación

## 4.1 · Optimización de profundidad (5 configuraciones)

In [ ]:
PROFUNDIDADES_DT = [3, 6, 10, 15, 20]
acc_tr_dt, acc_te_dt = [], []

for d in PROFUNDIDADES_DT:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    acc_tr_dt.append(accuracy_score(y_train, dt.predict(X_train)))
    acc_te_dt.append(accuracy_score(y_test,  dt.predict(X_test)))
    print(f'  max_depth={d:>2}  │  Train: {acc_tr_dt[-1]:.4f}   Test: {acc_te_dt[-1]:.4f}')

MEJOR_IDX_DT  = int(np.argmax(acc_te_dt))
MEJOR_PROF_DT = PROFUNDIDADES_DT[MEJOR_IDX_DT]
print(f'\n✔  Mejor profundidad: {MEJOR_PROF_DT}  (Test accuracy: {acc_te_dt[MEJOR_IDX_DT]:.4f})')

## 4.2 · Gráfica: Accuracy vs Profundidad

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(PROFUNDIDADES_DT, acc_tr_dt, 'o-', color='#9B8DC4', lw=2.2,
        markersize=9, label='Entrenamiento')
ax.plot(PROFUNDIDADES_DT, acc_te_dt, 's-', color='#2ECC71', lw=2.2,
        markersize=9, label='Prueba')
ax.axvline(MEJOR_PROF_DT, color='#E74C3C', ls='--', lw=1.6,
           label=f'Óptimo: depth={MEJOR_PROF_DT}')
ax.annotate(f' {acc_te_dt[MEJOR_IDX_DT]:.4f}',
            xy=(MEJOR_PROF_DT, acc_te_dt[MEJOR_IDX_DT]),
            color='#2ECC71', fontsize=11, fontweight='bold')

ax.set_title('Árbol de Decisión — Accuracy vs Profundidad', fontsize=13,
             fontweight='bold', pad=12)
ax.set_xlabel('max_depth', fontsize=11); ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(PROFUNDIDADES_DT)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.2f}'))
ax.legend(fontsize=10, facecolor='#1A1A1A', edgecolor='#444')
plt.tight_layout()
plt.savefig('grafica_arbol.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 4.3 · Predicciones con el modelo optimizado

In [ ]:
dt_opt = DecisionTreeClassifier(max_depth=MEJOR_PROF_DT, random_state=42)
dt_opt.fit(X_train, y_train)
y_pred_dt = dt_opt.predict(X_test)
ACC_DT = accuracy_score(y_test, y_pred_dt)

print(f'=== Árbol de Decisión — max_depth={MEJOR_PROF_DT} ===')
print(f'Accuracy: {ACC_DT:.4f}  ({ACC_DT*100:.2f} %)\n')
print(classification_report(y_test, y_pred_dt, target_names=CLASES))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
cm = confusion_matrix(y_test, y_pred_dt)
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(
    ax=ax, colorbar=True, cmap='Purples', xticks_rotation=45)
ax.set_facecolor('#0D0D0D')
ax.set_title(f'Matriz de Confusión — Árbol de Decisión (depth={MEJOR_PROF_DT})',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('cm_arbol.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

## 4.4 · Observaciones — Árbol de Decisión

**Sobre el comportamiento del modelo:**
- A profundidades **bajas (3–6)** el modelo **subajusta** (*underfitting*): tanto el accuracy de entrenamiento como el de prueba son bajos. El árbol carece de capacidad expresiva para separar 16 clases usando 60 variables.
- A profundidades **altas (15–20)** el accuracy de entrenamiento se aproxima a 1.0, indicando **sobreajuste** (*overfitting*); el modelo memoriza el conjunto de entrenamiento en lugar de generalizar.
- La mejor generalización se logra en `max_depth=15-20`, donde la brecha train/test se estabiliza.

**Limitaciones identificadas:**
- Alta varianza: pequeños cambios en los datos producen árboles completamente distintos.
- La interpretabilidad se pierde a profundidades mayores a 6–8.
- Un árbol único no aprovecha la riqueza del dataset de 60,000 registros.

<a id='bagging'></a>
---
# 🎒 PARTE V · Modelo de Bagging de Clasificación

## 5.1 · Optimización de profundidad del árbol base (5 configuraciones)

In [ ]:
PROFUNDIDADES_BAG = [5, 10, 15, 20, None]
ETIQ_BAG          = ['5', '10', '15', '20', 'Sin límite']
acc_tr_bag, acc_te_bag = [], []

for d, etiq in zip(PROFUNDIDADES_BAG, ETIQ_BAG):
    base = DecisionTreeClassifier(max_depth=d, random_state=42)
    bag  = BaggingClassifier(estimator=base, n_estimators=30,
                              max_samples=0.8, random_state=42, n_jobs=-1)
    bag.fit(X_train, y_train)
    acc_tr_bag.append(accuracy_score(y_train, bag.predict(X_train)))
    acc_te_bag.append(accuracy_score(y_test,  bag.predict(X_test)))
    print(f'  depth={etiq:<10} │  Train: {acc_tr_bag[-1]:.4f}   Test: {acc_te_bag[-1]:.4f}')

MEJOR_IDX_BAG  = int(np.argmax(acc_te_bag))
MEJOR_PROF_BAG = PROFUNDIDADES_BAG[MEJOR_IDX_BAG]
MEJOR_ETIQ_BAG = ETIQ_BAG[MEJOR_IDX_BAG]
print(f'\n✔  Mejor profundidad base: {MEJOR_ETIQ_BAG}  (Test accuracy: {acc_te_bag[MEJOR_IDX_BAG]:.4f})')

## 5.2 · Gráfica: Accuracy vs Profundidad del árbol base

In [ ]:
x_pos = list(range(len(ETIQ_BAG)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_pos, acc_tr_bag, 'o-', color='#9B8DC4', lw=2.2,
        markersize=9, label='Entrenamiento')
ax.plot(x_pos, acc_te_bag, 's-', color='#EF9A3A', lw=2.2,
        markersize=9, label='Prueba')
ax.axvline(MEJOR_IDX_BAG, color='#E74C3C', ls='--', lw=1.6,
           label=f'Óptimo: depth={MEJOR_ETIQ_BAG}')
ax.annotate(f' {acc_te_bag[MEJOR_IDX_BAG]:.4f}',
            xy=(MEJOR_IDX_BAG, acc_te_bag[MEJOR_IDX_BAG]),
            color='#EF9A3A', fontsize=11, fontweight='bold')

ax.set_title('Bagging — Accuracy vs Profundidad del Árbol Base', fontsize=13,
             fontweight='bold', pad=12)
ax.set_xlabel('Profundidad árbol base', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_xticks(x_pos); ax.set_xticklabels(ETIQ_BAG)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.2f}'))
ax.legend(fontsize=10, facecolor='#1A1A1A', edgecolor='#444')
plt.tight_layout()
plt.savefig('grafica_bagging.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 5.3 · Predicciones con el modelo optimizado

In [ ]:
base_opt = DecisionTreeClassifier(max_depth=MEJOR_PROF_BAG, random_state=42)
bag_opt  = BaggingClassifier(estimator=base_opt, n_estimators=30,
                              max_samples=0.8, random_state=42, n_jobs=-1)
bag_opt.fit(X_train, y_train)
y_pred_bag = bag_opt.predict(X_test)
ACC_BAG    = accuracy_score(y_test, y_pred_bag)

print(f'=== Bagging — base depth={MEJOR_ETIQ_BAG}, 30 estimadores ===')
print(f'Accuracy: {ACC_BAG:.4f}  ({ACC_BAG*100:.2f} %)\n')
print(classification_report(y_test, y_pred_bag, target_names=CLASES))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
cm_bag = confusion_matrix(y_test, y_pred_bag)
ConfusionMatrixDisplay(cm_bag, display_labels=CLASES).plot(
    ax=ax, colorbar=True, cmap='YlOrBr', xticks_rotation=45)
ax.set_facecolor('#0D0D0D')
ax.set_title(f'Matriz de Confusión — Bagging (base depth={MEJOR_ETIQ_BAG})',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('cm_bagging.png', dpi=150, bbox_inches='tight', facecolor='#0D0D0D')
plt.show()

## 5.4 · Observaciones — Bagging

**Mecanismo Bootstrap Aggregating:**
```
Dataset original → 30 submuestras (80% c/u, con reemplazo) → 30 árboles → voto por mayoría
```

**Comportamiento observado:**
- El Bagging **mejoró sustancialmente** el accuracy respecto al árbol individual al reducir la varianza mediante el promedio de 30 estimadores.
- Los árboles base **sin límite de profundidad** (`max_depth=None`) produjeron el mejor resultado: cada árbol puede especializarse y el promedio elimina el sobreajuste individual.
- La **brecha train/test es menor** que en el árbol simple, confirmando mejor generalización.
- La matriz de confusión muestra una diagonal más pronunciada que en el árbol individual.

<a id='rf'></a>
---
# 🌲 PARTE VI · Modelo Random Forest con GridSearchCV

## 6.1 · Definición del espacio de hiperparámetros

In [ ]:
param_grid = {
    'n_estimators'     : [50, 100, 200],
    'max_depth'        : [10, 20, None],
    'min_samples_split': [2, 5, 10],
}

n_combos = (len(param_grid['n_estimators']) *
            len(param_grid['max_depth']) *
            len(param_grid['min_samples_split']))

resumen_grid = pd.DataFrame([
    {'Hiperparámetro':'n_estimators',     'Valores probados':str(param_grid['n_estimators']),     'Descripción':'Número de árboles en el bosque'},
    {'Hiperparámetro':'max_depth',        'Valores probados':str(param_grid['max_depth']),        'Descripción':'Profundidad máxima de cada árbol'},
    {'Hiperparámetro':'min_samples_split','Valores probados':str(param_grid['min_samples_split']),'Descripción':'Mínimo de muestras para dividir un nodo'},
])

print(f'Total de combinaciones : {n_combos}')
print(f'Folds de cross-validation: 3')
print(f'Total de ajustes       : {n_combos * 3}\n')
resumen_grid

## 6.2 · Entrenamiento y validación con GridSearchCV

In [ ]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator          = rf,
    param_grid         = param_grid,
    cv                 = 3,
    scoring            = 'accuracy',
    n_jobs             = -1,
    verbose            = 1,
    return_train_score = True
)

print('⏳ Ejecutando GridSearchCV con 3-fold cross-validation...')
print('   (puede tardar varios minutos dependiendo del hardware)\n')
grid_search.fit(X_train, y_train)
print('\n✔  GridSearchCV completado')

## 6.3 · Selección del mejor modelo

In [ ]:
MEJOR_RF  = grid_search.best_estimator_
y_pred_rf = MEJOR_RF.predict(X_test)
ACC_RF    = accuracy_score(y_test, y_pred_rf)

print('╔══════════════════════════════════════════╗')
print('║      MEJORES HIPERPARÁMETROS             ║')
print('╠══════════════════════════════════════════╣')
for k, v in grid_search.best_params_.items():
    print(f'║  {k:<22}: {str(v):<16}║')
print('╠══════════════════════════════════════════╣')
print(f'║  CV accuracy (media) : {grid_search.best_score_:.4f}            ║')
print(f'║  Accuracy en prueba  : {ACC_RF:.4f}            ║')
print('╚══════════════════════════════════════════╝')

## 6.4 · Tabla de resultados del GridSearchCV (Top 10)

In [ ]:
res = pd.DataFrame(grid_search.cv_results_)
tabla_res = (res[['param_n_estimators','param_max_depth','param_min_samples_split',
                  'mean_train_score','mean_test_score','rank_test_score']]
             .rename(columns={
                 'param_n_estimators'     :'n_estimators',
                 'param_max_depth'        :'max_depth',
                 'param_min_samples_split':'min_samples_split',
                 'mean_train_score'       :'Acc Train (CV)',
                 'mean_test_score'        :'Acc Val (CV)',
                 'rank_test_score'        :'Ranking',
             })
             .sort_values('Ranking').reset_index(drop=True))

for col in ['Acc Train (CV)','Acc Val (CV)']:
    tabla_res[col] = tabla_res[col].round(4)

tabla_res.head(10).style.highlight_max(
    subset='Acc Val (CV)', color='#0D2B1A').hide(axis='index')

## 6.5 · Gráficas de resultados del GridSearchCV

In [ ]:
# ── Gráfica A: Heatmap n_estimators × max_depth (min_samples_split=2) ────────
pivot = res[res['param_min_samples_split']==2].pivot_table(
    index='param_max_depth', columns='param_n_estimators', values='mean_test_score')

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, cmap='Greens', aspect='auto',
               vmin=pivot.values.min()-0.003, vmax=pivot.values.max()+0.003)
plt.colorbar(im, ax=ax, label='Accuracy CV')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
ax.set_xlabel('n_estimators', fontsize=11)
ax.set_ylabel('max_depth', fontsize=11)
ax.set_title('GridSearchCV — Heatmap Accuracy CV\n(min_samples_split=2)',
             fontsize=12, fontweight='bold')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:.4f}", ha='center', va='center',
                fontsize=9.5, fontweight='bold', color='black')
plt.tight_layout()
plt.savefig('grid_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# ── Gráfica B: Accuracy CV vs n_estimators por min_samples_split ─────────────
colores_mss = ['#9B8DC4','#2ECC71','#EF9A3A']
marcadores  = ['o','s','^']

fig, ax = plt.subplots(figsize=(10, 5))
for idx, (mss, grp) in enumerate(res.groupby('param_min_samples_split')):
    sub = grp.groupby('param_n_estimators')['mean_test_score'].mean()
    ax.plot(sub.index, sub.values, marker=marcadores[idx], lw=2.2,
            markersize=9, color=colores_mss[idx],
            label=f'min_samples_split={mss}')

ax.set_title('GridSearchCV — Accuracy CV vs n_estimators', fontsize=13,
             fontweight='bold', pad=12)
ax.set_xlabel('n_estimators', fontsize=11)
ax.set_ylabel('Accuracy CV (media)', fontsize=11)
ax.set_xticks([50, 100, 200])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.3f}'))
ax.legend(fontsize=10, facecolor='#1A1A1A', edgecolor='#444')
plt.tight_layout()
plt.savefig('grid_estimadores.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 6.6 · Predicciones y evaluación del modelo optimizado

In [ ]:
print(f'=== Random Forest Optimizado — {grid_search.best_params_} ===')
print(f'Accuracy: {ACC_RF:.4f}  ({ACC_RF*100:.2f} %)\n')
print(classification_report(y_test, y_pred_rf, target_names=CLASES))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
cm_rf = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm_rf, display_labels=CLASES).plot(
    ax=ax, colorbar=True, cmap='Greens', xticks_rotation=45)
ax.set_facecolor('#0D0D0D')
ax.set_title('Matriz de Confusión — Random Forest (GridSearchCV)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('cm_rf.png', dpi=150, bbox_inches='tight', facecolor='#0D0D0D')
plt.show()

In [ ]:
# ── Top 15 variables más importantes ────────────────────────────────────────
imp = pd.Series(MEJOR_RF.feature_importances_,
                index=X.columns).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(range(len(imp)), imp.values, color='#3498DB',
               edgecolor='#0D0D0D', height=0.7)
ax.set_yticks(range(len(imp)))
ax.set_yticklabels([t[:60] for t in imp.index], fontsize=8)
for bar, val in zip(bars, imp.values):
    ax.text(val + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8, color='#EEEEEE')
ax.set_title('Random Forest — Top 15 Variables Más Importantes',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Importancia (Gini)', fontsize=11)
plt.tight_layout()
plt.savefig('importancias_rf.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 6.7 · Comparativa final de los tres modelos

In [ ]:
nombres = [
    f'Árbol de Decisión\n(depth={MEJOR_PROF_DT})',
    f'Bagging\n(base depth={MEJOR_ETIQ_BAG}, 30 est.)',
    f'Random Forest\n(GridSearchCV)',
]
valores  = [ACC_DT * 100, ACC_BAG * 100, ACC_RF * 100]
colores_ = ['#9B8DC4', '#EF9A3A', '#2ECC71']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(nombres, valores, color=colores_, edgecolor='#0D0D0D', height=0.5)
for bar, val in zip(bars, valores):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.2f} %', va='center', fontsize=12, fontweight='bold',
            color='#EEEEEE')
ax.set_xlim(0, 108)
ax.set_xlabel('Accuracy en Prueba (%)', fontsize=11)
ax.set_title('Comparativa de Modelos', fontsize=14, fontweight='bold', pad=12)
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.savefig('comparativa.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

comp_df = pd.DataFrame({'Modelo':nombres,'Accuracy (%)':[round(v,2) for v in valores]})
comp_df.style.highlight_max(subset='Accuracy (%)', color='#0D2B1A').hide(axis='index')

<a id='conclusiones'></a>
---
# 📝 PARTE VII · Análisis y Conclusiones

---
## 7.1 · Análisis del Dataset

| Característica | Detalle | Implicación |
|---|---|---|
| Registros | 59,999 | Dataset grande → modelos robustos |
| Variables (features) | 60 preguntas Likert | Alta dimensionalidad manejable |
| Clases objetivo | 16 tipos MBTI | Clasificación multiclase compleja |
| Balance de clases | ~3,740–3,769 por clase | Distribución casi perfecta |
| Tipo de escala | Valores numéricos continuos | Compatible con árboles directamente |

> **Observación clave:** El balance casi perfecto entre las 16 clases (~3,750 registros cada una) eliminó la necesidad de técnicas de rebalanceo como SMOTE o ponderación de clases, simplificando el pipeline.

---
## 7.2 · Análisis Comparativo de Modelos

### Árbol de Decisión
- **Fortaleza:** Interpretable, rápido, no requiere normalización de datos.
- **Debilidad:** Alta varianza. Un árbol único es sensible a pequeñas variaciones en los datos de entrenamiento.
- **Fenómeno observado:** La curva de accuracy de prueba se estabiliza a grandes profundidades, mientras la de entrenamiento sigue subiendo → sobreajuste progresivo.

### Bagging
- **Fortaleza:** Redujo la varianza del árbol individual al promediar 30 estimadores entrenados en submuestras distintas.
- **Mecanismo:** $\hat{f}_{bag}(x) = \frac{1}{B}\sum_{b=1}^{B} \hat{f}_b(x)$
- **Resultado:** Mejora significativa de accuracy respecto al árbol simple sin aumentar el sesgo.

### Random Forest
- **Fortaleza:** Añade aleatoriedad en la selección de features por nodo, reduciendo la correlación entre árboles.
- **GridSearchCV** permitió encontrar la combinación óptima de hiperparámetros de forma sistemática y reproducible, evitando el sesgo de la selección manual.
- **Resultado:** Mayor accuracy en prueba entre los tres modelos.

---
## 7.3 · Análisis de Variables Importantes

El análisis de `feature_importances_` del Random Forest reveló que:

- Las preguntas de mayor importancia tienden a medir directamente las **dimensiones MBTI** más discriminativas, especialmente **Introversión/Extraversión (E/I)** y **Pensamiento/Sentimiento (T/F)**.
- Las preguntas relacionadas con **preferencias sociales** (eventos grupales, llamadas telefónicas, hacer amigos) son altamente informativas para el modelo.
- Existe un **20% de preguntas** que aportan muy poca información discriminativa — candidatas a eliminación en una etapa de reducción de dimensionalidad.

---
## 7.4 · Conclusiones Finales

### ✅ Logros del proyecto

1. Se construyó un **pipeline completo** de clasificación supervisada: carga → traducción → preprocesamiento → modelado → evaluación → comparación.
2. Se implementaron y compararon **3 modelos de complejidad creciente**, demostrando empíricamente cómo el ensamble mejora la generalización.
3. La optimización con **GridSearchCV + cross-validation** garantizó una selección de hiperparámetros objetiva, reproducible y sin fuga de datos.
4. El **Random Forest optimizado** fue el modelo más adecuado para este problema de clasificación multiclase.

### 🚀 Posibles mejoras futuras

| Mejora | Justificación |
|---|---|
| **Gradient Boosting (XGBoost / LightGBM)** | Captura relaciones no lineales más complejas con menor número de árboles |
| **Reducción de dimensionalidad (PCA / SelectKBest)** | El análisis de importancias sugiere que 10-15 preguntas podrían ser suficientes |
| **Redes neuronales (MLP)** | Podría superar al RF capturando interacciones de alto orden entre preguntas |
| **Ampliar el grid de hiperparámetros** | Incluir `max_features`, `min_samples_leaf` y `bootstrap` en la búsqueda |
| **Validación con datos reales** | El dataset parece sintético; aplicar el modelo a respuestas reales validaría su utilidad práctica |

### 💡 Reflexión final

> El ejercicio demuestra que la elección del modelo no puede desvincularse del contexto del problema. Para clasificación multiclase con datos tabulares y clases balanceadas, los modelos de ensamble basados en árboles son una elección sólida y eficiente. La inversión en la optimización sistemática de hiperparámetros con GridSearchCV se traduce directamente en mejoras medibles de precisión, y el análisis de importancia de variables agrega valor interpretativo al modelo.

---
*Proyecto Final — Modelos de Clasificación en Machine Learning*